[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/ai-foundations/blob/main/03_thinking_in_arrays/tutorial.ipynb)

## ▶️ Run this notebook in Google Colab

**Option A — One click (recommended):** Click the **Open in Colab** badge above. It opens this notebook directly in a free cloud runtime — nothing to install locally.

**Option B — Upload manually:**
1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this `tutorial.ipynb`.

**Then, in Colab:**
1. Run the **first code cell** (the dependency-setup cell) — it installs everything this notebook needs directly into the Colab kernel. No `pip install`, `uv sync`, or `pyproject.toml` required.
2. Run the remaining cells top to bottom via **Runtime → Run all** (or `Ctrl/Cmd+F9`).

> 💡 This notebook is **fully self-contained**: any data files it uses are generated inside the notebook, so it runs end-to-end in Colab with no external files.

---

In [ ]:
# === Inline dependency setup (self-contained; no `uv sync` / pyproject.toml needed) ===
# Installs this notebook's libraries directly into the running kernel.
# Works in local Jupyter, VS Code, and Google Colab. Safe to re-run (idempotent).
import sys, subprocess

_DEPS = [
    'python-dotenv>=1.0.0',
    'numpy>=1.26.0',
    'matplotlib>=3.7',
]

# Ensure pip is available in this kernel (some minimal venvs ship without it), then install.
try:
    import pip  # noqa: F401
except ModuleNotFoundError:
    import ensurepip; ensurepip.bootstrap()

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_DEPS])
print('✓ Dependencies installed:', ', '.join(_DEPS))

In [ ]:
# === Google Colab: native Mermaid diagram rendering ===
# Colab does not render ```mermaid Markdown blocks. This helper injects the
# official Mermaid.js library into a cell's output and draws the diagram there.
# The Markdown cells keep their ```mermaid blocks so GitHub still renders them;
# the code cells below each diagram call render_mermaid(...) so Colab does too.
# Safe to re-run.
import IPython.display as _ipd

_MERMAID_COUNTER = 0


def render_mermaid(graph: str):
    # Render a Mermaid diagram in the current cell's output (works in Colab).
    global _MERMAID_COUNTER
    _MERMAID_COUNTER += 1
    container_id = f"mermaid-diagram-{_MERMAID_COUNTER}"
    # The graph text is placed verbatim inside the .mermaid div; Mermaid reads it
    # as textContent, so it must NOT be HTML-escaped.
    html = f'''
<div id="{container_id}" class="mermaid">
{graph}
</div>
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
  mermaid.initialize({{ startOnLoad: false }});
  const el = document.getElementById("{container_id}");
  await mermaid.run({{ nodes: [el] }});
</script>
'''
    _ipd.display(_ipd.HTML(html))


# Module 3: Thinking in Arrays (Tensors)

### From Python Lists to NumPy Arrays — Building the Foundation for AI


This notebook teaches you how to **think in arrays** — the fundamental shift from Python's list-based thinking to NumPy's contiguous-memory, vectorized approach. By the end you will understand why Python lists fail for AI workloads, how NumPy arrays work under the hood, and how to convert discrete tokens (words) into the numerical matrices that neural networks actually process.

We build toward one concrete goal: taking the tokenizer output from **Module 1** and transforming it into a **one-hot encoded matrix** — the classical first step before embeddings.

## 📋 Summary: the one-paragraph version

Python lists are **scattered across memory** like books on different shelves — every access requires looking up a pointer, which kills performance when you need to do math on millions of numbers. **NumPy arrays** store data in a **contiguous block** like a solid brick of numbers, enabling the CPU to process them in parallel with vectorized operations that are **50-100x faster**. But neural networks cannot read words — they read numbers. **One-hot encoding** converts each word into a sparse binary vector (one `1`, rest `0`s), stacking them into a matrix that becomes the **input** to a neural network. This sparsity problem (99.9% zeros) is why we later move to **dense embeddings**.

> ### 🧮 The mental model
>
> **List thinking**: "I have a collection of things"
>
> **Array thinking**: "I have a rectangular block of numbers with shape"
>
> That shape — **(rows, columns)** or **(depth, rows, columns)** — is the key to everything.

## 🗺️ What you will learn, step by step

| Step | What you do | Key concept |
| ---: | --- | --- |
| 0 | Set up the environment | `numpy`, `matplotlib`, `python-dotenv` |
| 1 | **Why Python lists fail** | Non-contiguous memory, pointer overhead, cache misses |
| 2 | **Create NumPy arrays** | `np.array()`, `np.zeros()`, `np.ones()`, `np.arange()`, `np.random` |
| 3 | **Understand .shape** | Reading dimensions: `(5,)` vs `(5,1)` vs `(3,4)` vs `(2,3,4)` |
| 4 | **Rank-1 vs Rank-2 arrays** | The shape bug that breaks matrix operations |
| 5 | **Reshape arrays** | Transforming dimensions without copying: `.reshape()`, `.flatten()` |
| 6 | **Index and slice** | Accessing elements: `arr[row, col]`, boolean masking |
| 7 | **One-hot encoding** | Words → vocabulary → index map → binary vectors → matrix |
| 8 | **Lab: The Matrix** | Build a one-hot encoder from scratch |

### ✅ Prerequisites

- Basic Python (lists, loops, dictionaries)
- **Module 1** (tokenization) — we reuse the vocabulary concept here
- No linear algebra required (we explain shape intuitively)

---
# Step 0 · Setup

## 0.1 Install the libraries

We use **NumPy** for arrays and **matplotlib** for visualization. The dependency list lives in `pyproject.toml`:

```toml
[project]
requires-python = ">=3.10"
dependencies = [
    "numpy>=1.24",
    "matplotlib>=3.7",
    "python-dotenv",
    "jupyterlab",
    "ipykernel",
]
```

**Recommended setup with `uv`** (from the module directory):

```bash
uv sync
uv run jupyter lab
```

If you are on Colab or another hosted environment without `uv`, run the cell below instead.

## 0.2 Load environment variables

We follow the same `.env` pattern from Module 1 to keep any future API keys out of the notebook.

In [ ]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
print("Environment loaded.")

## 0.3 Imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple

%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print("All imports succeeded.")

---
# Step 1 · Why Python Lists Fail for AI

Before we introduce NumPy, we need to understand **why Python's built-in lists are too slow** for numerical AI work. The problem is **memory layout**.

## 1.1 Python lists: scattered pointers

A Python list does not store the actual numbers — it stores **pointers** to Python objects scattered across memory. Each element is a full `PyObject` with type information, reference count, and other metadata.

```mermaid
graph LR
    A["Python List<br/>[ref, ref, ref, ref]"]
    B["PyObject<br/>value=1"]
    C["PyObject<br/>value=2"]
    D["PyObject<br/>value=3"]
    E["PyObject<br/>value=4"]
    A -->|pointer 1| B
    A -->|pointer 2| C
    A -->|pointer 3| D
    A -->|pointer 4| E
    
    style A fill:#e94560,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#fafafa,stroke:#333,stroke-width:1px
    style C fill:#fafafa,stroke:#333,stroke-width:1px
    style D fill:#fafafa,stroke:#333,stroke-width:1px
    style E fill:#fafafa,stroke:#333,stroke-width:1px
```

**Problems:**
1. **Non-contiguous memory** — each number lives at a random address
2. **Python object overhead** — every integer carries ~28 bytes of metadata
3. **CPU cache misses** — accessing the next element requires chasing a pointer
4. **No vectorization** — the CPU cannot process multiple elements in parallel

## 1.2 NumPy arrays: contiguous blocks

A NumPy array stores raw numeric values in a **contiguous block of memory** — one solid chunk, no pointers.

```mermaid
graph LR
    A["NumPy Array (ndarray)<br/><br/>[1][2][3][4]"]
    
    style A fill:#16a085,stroke:#333,stroke-width:2px,color:#fff
```

**Advantages:**
1. **Contiguous memory** — all numbers sit next to each other in RAM
2. **Fixed dtype** — no per-element overhead (e.g., `int64` = 8 bytes per number)
3. **CPU cache friendly** — fetching one element loads neighbors into cache
4. **Vectorization** — CPU SIMD instructions process 4-8 numbers per cycle

> ### 🚀 Performance difference: 50-100x faster
>
> For operations like `sum`, `mean`, or element-wise math, NumPy is **50-100 times faster** than Python lists — not because NumPy is written in C (which helps), but because **contiguous memory enables vectorization**.

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph LR
    A["Python List<br/>[ref, ref, ref, ref]"]
    B["PyObject<br/>value=1"]
    C["PyObject<br/>value=2"]
    D["PyObject<br/>value=3"]
    E["PyObject<br/>value=4"]
    A -->|pointer 1| B
    A -->|pointer 2| C
    A -->|pointer 3| D
    A -->|pointer 4| E
    
    style A fill:#e94560,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#fafafa,stroke:#333,stroke-width:1px
    style C fill:#fafafa,stroke:#333,stroke-width:1px
    style D fill:#fafafa,stroke:#333,stroke-width:1px
    style E fill:#fafafa,stroke:#333,stroke-width:1px
''')
render_mermaid(r'''
graph LR
    A["NumPy Array (ndarray)<br/><br/>[1][2][3][4]"]
    
    style A fill:#16a085,stroke:#333,stroke-width:2px,color:#fff
''')


Let's prove it with a simple benchmark:

In [ ]:
import time

# Create a large list and a large NumPy array
size = 1_000_000
python_list = list(range(size))
numpy_array = np.arange(size)

# Benchmark: sum all elements (Python list)
start = time.perf_counter()
list_sum = sum(python_list)
list_time = time.perf_counter() - start

# Benchmark: sum all elements (NumPy array)
start = time.perf_counter()
array_sum = np.sum(numpy_array)
array_time = time.perf_counter() - start

print(f"Python list sum: {list_time*1000:.2f} ms")
print(f"NumPy array sum: {array_time*1000:.2f} ms")
print(f"\nSpeedup: {list_time / array_time:.1f}x faster")
print(f"(Both results equal: {list_sum == array_sum})")

> ### 💡 Key Insight
>
> **NumPy is not "just faster C code."** It is faster because it stores data the way the CPU wants to see it: **contiguous, typed, and cache-friendly**. This is the foundational idea that makes modern AI possible — when you are doing math on 175 billion parameters, every millisecond per operation multiplies into days of training time.

---
# Step 2 · Creating NumPy Arrays

Now that we know *why* NumPy exists, let's learn *how* to create arrays. There are many ways — we will cover the six most useful.

## 2.1 From Python lists: `np.array()`

The most direct way — convert an existing Python list.

In [ ]:
# Create from a Python list
python_list = [1, 2, 3, 4, 5]
arr = np.array(python_list)

print("Python list:", python_list)
print("NumPy array:", arr)
print("Type:       ", type(arr))
print("dtype:      ", arr.dtype)  # int64 on most systems

**What is `dtype`?** It is the **data type** of every element in the array. Unlike Python lists (which can mix types), **every element in a NumPy array has the same dtype**. Common dtypes:

| dtype | Meaning | Bytes per element |
|-------|---------|------------------|
| `int32` | 32-bit integer | 4 |
| `int64` | 64-bit integer | 8 |
| `float32` | 32-bit float | 4 |
| `float64` | 64-bit float | 8 |
| `bool` | Boolean | 1 |

You can specify dtype explicitly:

In [ ]:
# Force float32 (useful for neural networks to save memory)
arr_float32 = np.array([1, 2, 3], dtype=np.float32)
print("Array:     ", arr_float32)
print("dtype:     ", arr_float32.dtype)
print("Memory:    ", arr_float32.nbytes, "bytes")

## 2.2 Arrays of zeros: `np.zeros()`

Often you need to preallocate an array and fill it later. `np.zeros()` creates an array of all zeros.

In [ ]:
# 1D array of 5 zeros
zeros_1d = np.zeros(5)
print("1D zeros:", zeros_1d)
print("dtype:   ", zeros_1d.dtype)  # float64 by default

# 2D array (3 rows, 4 columns) of zeros
zeros_2d = np.zeros((3, 4))
print("\n2D zeros (3x4):")
print(zeros_2d)

## 2.3 Arrays of ones: `np.ones()`

Same idea, but filled with ones.

In [ ]:
ones = np.ones((2, 3), dtype=np.int32)
print("2D ones (2x3):")
print(ones)

## 2.4 Ranges: `np.arange()`

Like Python's `range()`, but returns an array.

In [ ]:
# np.arange(start, stop, step)
range_arr = np.arange(0, 10, 2)
print("np.arange(0, 10, 2):", range_arr)

# Default start=0, step=1
simple_range = np.arange(5)
print("np.arange(5):       ", simple_range)

## 2.5 Evenly spaced: `np.linspace()`

When you need **N evenly spaced points** between two values (useful for plotting).

In [ ]:
# 5 points from 0 to 1 (inclusive)
linspace_arr = np.linspace(0, 1, 5)
print("np.linspace(0, 1, 5):", linspace_arr)

## 2.6 Random arrays

For testing, initialization, and data augmentation.

In [ ]:
# Random floats in [0, 1)
rand_uniform = np.random.rand(3, 2)
print("np.random.rand(3, 2):")
print(rand_uniform)

# Random from standard normal distribution (mean=0, std=1)
rand_normal = np.random.randn(4)
print("\nnp.random.randn(4):", rand_normal)

# Random integers
rand_ints = np.random.randint(0, 10, size=5)
print("\nnp.random.randint(0, 10, size=5):", rand_ints)

---
# Step 3 · The .shape Property

**The most important property of a NumPy array is `.shape`** — a tuple that tells you the size of each dimension.

```mermaid
graph TD
    A["1D Array: shape = (5,)<br/>[1, 2, 3, 4, 5]"]
    B["2D Array: shape = (3, 4)<br/>3 rows, 4 columns"]
    C["3D Array: shape = (2, 3, 4)<br/>2 layers, 3 rows, 4 columns"]
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
```

## 3.1 One-dimensional (1D) arrays

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph TD
    A["1D Array: shape = (5,)<br/>[1, 2, 3, 4, 5]"]
    B["2D Array: shape = (3, 4)<br/>3 rows, 4 columns"]
    C["3D Array: shape = (2, 3, 4)<br/>2 layers, 3 rows, 4 columns"]
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
''')


In [ ]:
arr_1d = np.array([1, 2, 3, 4, 5])

print("Array:     ", arr_1d)
print("Shape:     ", arr_1d.shape)     # (5,)
print("Dimensions:", arr_1d.ndim)      # 1
print("Size:      ", arr_1d.size)      # 5 (total elements)

**Reading the shape:**
- `shape = (5,)` means **5 elements in one dimension**
- The trailing comma is Python's way of saying "this is a tuple with one element"
- Think of this as a **vector** (in linear algebra terms)

## 3.2 Two-dimensional (2D) arrays

In [ ]:
arr_2d = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])

print("Array:")
print(arr_2d)
print("\nShape:     ", arr_2d.shape)    # (3, 4)
print("Dimensions:", arr_2d.ndim)       # 2
print("Size:      ", arr_2d.size)       # 12 (3 * 4)

**Reading the shape:**
- `shape = (3, 4)` means **3 rows, 4 columns**
- Think of this as a **matrix** or **table**
- Convention: `shape[0]` = rows, `shape[1]` = columns

## 3.3 Three-dimensional (3D) arrays

In [ ]:
# Create a 3D array: 2 layers, 3 rows, 4 columns
arr_3d = np.zeros((2, 3, 4))

print("Shape:     ", arr_3d.shape)      # (2, 3, 4)
print("Dimensions:", arr_3d.ndim)       # 3
print("Size:      ", arr_3d.size)       # 24 (2 * 3 * 4)
print("\nFirst layer:")
print(arr_3d[0])

**Reading the shape:**
- `shape = (2, 3, 4)` means **2 layers (or slices), each with 3 rows and 4 columns**
- Think of this as a **stack of matrices** or a **video frame** (batch_size, height, width)
- Common in AI: `(batch, sequence_length, embedding_dim)` for text, or `(batch, height, width, channels)` for images

## 3.4 Visualizing dimensions

Let's create a visual representation:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# 1D array visualization
arr_1d_vis = np.arange(5)
axes[0].imshow(arr_1d_vis.reshape(1, -1), cmap='Blues', aspect='auto')
axes[0].set_title('1D Array\nshape = (5,)', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(5))
axes[0].set_yticks([])
axes[0].set_xlabel('index')

# 2D array visualization
arr_2d_vis = np.arange(12).reshape(3, 4)
axes[1].imshow(arr_2d_vis, cmap='Purples', aspect='auto')
axes[1].set_title('2D Array\nshape = (3, 4)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('columns')
axes[1].set_ylabel('rows')

# 3D array visualization (show first layer)
arr_3d_vis = np.arange(24).reshape(2, 3, 4)
axes[2].imshow(arr_3d_vis[0], cmap='Reds', aspect='auto')
axes[2].set_title('3D Array (first layer)\nshape = (2, 3, 4)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('columns')
axes[2].set_ylabel('rows')

plt.tight_layout()
plt.show()

> ### 💡 Key Insight
>
> **Shape is the signature of your data.** When you get a dimension mismatch error (the most common NumPy error), it is because the shapes do not line up. Always check `.shape` first.

---
# Step 4 · Rank-1 vs Rank-2 Arrays (The Shape Bug)

**This is the most common mistake beginners make with NumPy.** Let's illustrate it with a concrete example.

## 4.1 The three shapes that look similar but are NOT

Consider these three arrays:

In [ ]:
a = np.array([1, 2, 3, 4, 5])           # shape = (5,)    — rank-1 array
b = np.array([[1, 2, 3, 4, 5]])         # shape = (1, 5)  — row vector
c = np.array([[1], [2], [3], [4], [5]]) # shape = (5, 1)  — column vector

print("a.shape:", a.shape, "\t", a)
print("b.shape:", b.shape, "\t", b)
print("c.shape:", c.shape, "\n", c)

**They all hold the same numbers, but they behave completely differently!**

```mermaid
graph TB
    A["a: shape = (5,)<br/>rank-1 array<br/>[1 2 3 4 5]"]
    B["b: shape = (1, 5)<br/>row vector<br/>[[1 2 3 4 5]]"]
    C["c: shape = (5, 1)<br/>column vector<br/>[[1]<br/> [2]<br/> [3]<br/> [4]<br/> [5]]"]
    
    style A fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#2ecc71,stroke:#333,stroke-width:2px,color:#fff
```

## 4.2 Why this matters: matrix operations

Let's try to compute the **dot product** (which requires matching dimensions):

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph TB
    A["a: shape = (5,)<br/>rank-1 array<br/>[1 2 3 4 5]"]
    B["b: shape = (1, 5)<br/>row vector<br/>[[1 2 3 4 5]]"]
    C["c: shape = (5, 1)<br/>column vector<br/>[[1]<br/> [2]<br/> [3]<br/> [4]<br/> [5]]"]
    
    style A fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#2ecc71,stroke:#333,stroke-width:2px,color:#fff
''')


In [ ]:
# This works:
print("np.dot(a, a):", np.dot(a, a))  # 1*1 + 2*2 + 3*3 + 4*4 + 5*5 = 55

# This fails (shape mismatch):
try:
    result = np.dot(b, b)  # (1, 5) @ (1, 5) — invalid!
except ValueError as e:
    print(f"\nError with np.dot(b, b): {e}")

# This works (transpose to match):
print("\nnp.dot(b, c):", np.dot(b, c))  # (1, 5) @ (5, 1) = (1, 1)

## 4.3 Fixing the shape bug with `.reshape()`

When you get a shape error, **reshape** is your friend:

In [ ]:
# Convert rank-1 to column vector
a_col = a.reshape(-1, 1)  # -1 means "infer this dimension"
print("a_col.shape:", a_col.shape)
print(a_col)

# Convert rank-1 to row vector
a_row = a.reshape(1, -1)
print("\na_row.shape:", a_row.shape)
print(a_row)

## 4.4 Using `np.newaxis` / `np.expand_dims`

Another way to add dimensions:

In [ ]:
# Using np.newaxis (same as None)
a_col_2 = a[:, np.newaxis]
print("a[:, np.newaxis].shape:", a_col_2.shape)

# Using np.expand_dims
a_row_2 = np.expand_dims(a, axis=0)
print("np.expand_dims(a, axis=0).shape:", a_row_2.shape)

> ### ⚠️ Caution: Always check your shapes
>
> Before any matrix operation, **print the shapes**. Most NumPy errors trace back to this:
> ```python
> print("A.shape:", A.shape, "B.shape:", B.shape)
> ```
> Debugging "shape = (5,) vs (5,1)" can save you hours.

---
# Step 5 · Reshaping Arrays

Reshaping transforms array dimensions **without copying data** — it is just a different view of the same memory.

## 5.1 Basic reshape

In [ ]:
# Start with 1D array of 12 elements
arr = np.arange(12)
print("Original:", arr.shape, arr)

# Reshape to 3 rows, 4 columns
reshaped = arr.reshape(3, 4)
print("\nReshaped to (3, 4):")
print(reshaped)

# Reshape to 2 layers, 2 rows, 3 columns
reshaped_3d = arr.reshape(2, 2, 3)
print("\nReshaped to (2, 2, 3):")
print(reshaped_3d)

## 5.2 Using -1 to infer dimension

NumPy can **automatically calculate one dimension** if you give it `-1`:

In [ ]:
arr = np.arange(24)

# "Make it 4 columns, figure out the rows"
auto_rows = arr.reshape(-1, 4)
print("reshape(-1, 4):", auto_rows.shape)

# "Make it 3 rows, figure out the columns"
auto_cols = arr.reshape(3, -1)
print("reshape(3, -1):", auto_cols.shape)

## 5.3 Flattening arrays

To convert any array back to 1D:

In [ ]:
arr_2d = np.array([[1, 2, 3], [4, 5, 6]])

# Method 1: .flatten() (makes a copy)
flat1 = arr_2d.flatten()
print(".flatten():", flat1)

# Method 2: .ravel() (returns a view if possible, no copy)
flat2 = arr_2d.ravel()
print(".ravel():  ", flat2)

# Method 3: .reshape(-1)
flat3 = arr_2d.reshape(-1)
print(".reshape(-1):", flat3)

## 5.4 Practical example: reshaping image data

Images are often stored as 3D arrays `(height, width, channels)`. Neural networks often want them flattened:

In [ ]:
# Simulate a 28x28 grayscale image (like MNIST digits)
image = np.random.rand(28, 28)
print("Image shape:", image.shape)

# Flatten for a neural network input layer
flattened = image.reshape(-1)  # or image.flatten()
print("Flattened:", flattened.shape)  # (784,)
print(f"Total pixels: {28*28} = {flattened.shape[0]}")

> ### 💡 Key Insight
>
> **Reshaping is free** (no computation) because NumPy just changes how it interprets the same memory. This is why you can experiment with different shapes without worrying about performance.

---
# Step 6 · Indexing and Slicing

NumPy extends Python's slicing syntax to multiple dimensions.

## 6.1 Basic indexing (1D)

In [ ]:
arr = np.array([10, 20, 30, 40, 50])

print("arr[0]:   ", arr[0])      # First element
print("arr[-1]:  ", arr[-1])     # Last element
print("arr[1:4]: ", arr[1:4])    # Slice from index 1 to 3 (4 is excluded)
print("arr[::2]: ", arr[::2])    # Every second element

## 6.2 Multi-dimensional indexing

Use `arr[row, col]` syntax:

In [ ]:
arr_2d = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])

print("Array:")
print(arr_2d)

print("\narr_2d[0, 0]:  ", arr_2d[0, 0])      # Top-left element (1)
print("arr_2d[1, 2]:  ", arr_2d[1, 2])        # Row 1, col 2 (7)
print("arr_2d[-1, -1]:", arr_2d[-1, -1])      # Bottom-right (12)

# Slice entire rows or columns
print("\narr_2d[1, :]:  ", arr_2d[1, :])      # Second row (all columns)
print("arr_2d[:, 2]:  ", arr_2d[:, 2])        # Third column (all rows)

## 6.3 Boolean indexing (masking)

One of NumPy's most powerful features — select elements that meet a condition:

In [ ]:
arr = np.array([1, 5, 10, 15, 20, 25])

# Create a boolean mask
mask = arr > 10
print("Mask (arr > 10):", mask)

# Apply the mask
filtered = arr[mask]
print("Filtered values:", filtered)

# One-liner
print("\nValues > 10:", arr[arr > 10])

This is how you filter datasets without loops:

```python
# Instead of:
result = []
for x in arr:
    if x > 10:
        result.append(x)

# Do this:
result = arr[arr > 10]
```

---
# Step 7 · One-Hot Encoding

Now we bring everything together. **Neural networks cannot read words** — they read numbers. One-hot encoding is the classical way to convert discrete tokens (words) into numerical vectors.

## 7.1 What is one-hot encoding?

**One-hot encoding** converts each word into a **sparse binary vector**: one element is `1` (hot), all others are `0` (cold).

```mermaid
graph LR
    A["Words"] --> B["Vocabulary"]
    B --> C["Index Map"]
    C --> D["One-Hot Vectors"]
    D --> E["Matrix"]
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
    style D fill:#f39c12,stroke:#333,stroke-width:2px,color:#fff
    style E fill:#16a085,stroke:#333,stroke-width:2px,color:#fff
```

**Example:**

| Word | Index | One-Hot Vector (vocab size = 5) |
|------|-------|----------------------------------|
| "the" | 0 | `[1, 0, 0, 0, 0]` |
| "cat" | 1 | `[0, 1, 0, 0, 0]` |
| "sat" | 2 | `[0, 0, 1, 0, 0]` |
| "on" | 3 | `[0, 0, 0, 1, 0]` |
| "mat" | 4 | `[0, 0, 0, 0, 1]` |

## 7.2 Why do we need it?

You cannot do math with words:

```python
"cat" + "dog" = ?  # Doesn't make sense
```

But you **can** do math with vectors:

```python
[0, 1, 0, 0, 0] + [0, 0, 0, 1, 0] = [0, 1, 0, 1, 0]  # Valid!
```

One-hot encoding is the **bridge from text to numbers**.

## 7.3 Building a vocabulary

First, we need a vocabulary — the unique words in our corpus:

In [ ]:
# Colab-native rendering of the Mermaid diagram(s) in the cell above.
# (The Markdown block still renders natively on GitHub.)
render_mermaid(r'''
graph LR
    A["Words"] --> B["Vocabulary"]
    B --> C["Index Map"]
    C --> D["One-Hot Vectors"]
    D --> E["Matrix"]
    
    style A fill:#3498db,stroke:#333,stroke-width:2px,color:#fff
    style B fill:#9b59b6,stroke:#333,stroke-width:2px,color:#fff
    style C fill:#e74c3c,stroke:#333,stroke-width:2px,color:#fff
    style D fill:#f39c12,stroke:#333,stroke-width:2px,color:#fff
    style E fill:#16a085,stroke:#333,stroke-width:2px,color:#fff
''')


In [ ]:
# Sample corpus (from Module 1)
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are animals"
]

# Tokenize (simple split)
tokens = [word.lower() for sentence in corpus for word in sentence.split()]
print("Tokens:", tokens)

# Build vocabulary (unique words)
vocab = sorted(set(tokens))
vocab_size = len(vocab)

print(f"\nVocabulary ({vocab_size} words):")
print(vocab)

## 7.4 Creating the word-to-index mapping

In [ ]:
# Create word → index mapping
word_to_idx = {word: idx for idx, word in enumerate(vocab)}

print("Word-to-Index Mapping:")
for word, idx in list(word_to_idx.items())[:5]:  # Show first 5
    print(f"  '{word}' → {idx}")

## 7.5 Creating one-hot vectors

Now we convert words to one-hot vectors:

In [ ]:
def word_to_onehot(word: str, word_to_idx: Dict[str, int], vocab_size: int) -> np.ndarray:
    """
    Convert a word to its one-hot vector representation.
    
    Args:
        word: The word to encode
        word_to_idx: Vocabulary mapping
        vocab_size: Size of vocabulary
        
    Returns:
        One-hot vector of shape (vocab_size,)
    """
    vector = np.zeros(vocab_size)
    if word in word_to_idx:
        idx = word_to_idx[word]
        vector[idx] = 1
    return vector

# Test with "cat"
cat_vector = word_to_onehot("cat", word_to_idx, vocab_size)
print(f"'cat' → one-hot: {cat_vector}")
print(f"Shape: {cat_vector.shape}")

# Test with "dog"
dog_vector = word_to_onehot("dog", word_to_idx, vocab_size)
print(f"\n'dog' → one-hot: {dog_vector}")

## 7.6 Stacking into a matrix

To encode a sentence, we stack one-hot vectors into a **matrix**:

In [ ]:
def sentence_to_onehot_matrix(sentence: str, word_to_idx: Dict[str, int], vocab_size: int) -> np.ndarray:
    """
    Convert a sentence to a one-hot encoded matrix.
    
    Args:
        sentence: Input sentence
        word_to_idx: Vocabulary mapping
        vocab_size: Size of vocabulary
        
    Returns:
        Matrix of shape (num_words, vocab_size)
    """
    words = sentence.lower().split()
    vectors = [word_to_onehot(word, word_to_idx, vocab_size) for word in words]
    return np.vstack(vectors)

# Encode "the cat sat"
sentence = "the cat sat"
matrix = sentence_to_onehot_matrix(sentence, word_to_idx, vocab_size)

print(f"Sentence: '{sentence}'")
print(f"Matrix shape: {matrix.shape}")
print("\nOne-Hot Matrix:")
print(matrix)

**Reading the matrix:**
- Each **row** is one word
- Each **column** is one vocabulary position
- Shape is `(num_words, vocab_size)`

## 7.7 Visualizing the sparsity

Let's visualize the matrix as a heatmap:

In [ ]:
# Encode the full first sentence
full_sentence = corpus[0]
full_matrix = sentence_to_onehot_matrix(full_sentence, word_to_idx, vocab_size)

plt.figure(figsize=(10, 6))
plt.imshow(full_matrix, cmap='RdYlGn', aspect='auto', interpolation='nearest')
plt.colorbar(label='Value')
plt.title(f"One-Hot Encoded Matrix: '{full_sentence}'\nShape: {full_matrix.shape}", 
          fontsize=14, fontweight='bold')
plt.xlabel('Vocabulary Index', fontsize=12)
plt.ylabel('Word Position in Sentence', fontsize=12)
plt.xticks(range(vocab_size), vocab, rotation=90, fontsize=8)
plt.yticks(range(len(full_sentence.split())), full_sentence.split())
plt.tight_layout()
plt.show()

# Calculate sparsity
total_elements = full_matrix.size
nonzero_elements = np.count_nonzero(full_matrix)
sparsity = (1 - nonzero_elements / total_elements) * 100

print(f"\nTotal elements: {total_elements}")
print(f"Non-zero elements: {nonzero_elements}")
print(f"Sparsity: {sparsity:.2f}%")

> ### ⚠️ The Sparsity Problem
>
> **One-hot vectors are 99%+ zeros.** For a vocabulary of 50,000 words, each word is a 50,000-dimensional vector with only **one** element set to 1. This is:
> - **Memory inefficient** (storing mostly zeros)
> - **Semantically weak** (no notion of word similarity: "cat" and "dog" are equally far)
>
> This is why modern NLP uses **dense embeddings** (Module 4) — vectors where every dimension carries meaning, not just a single index.

---
# Step 8 · Lab: The Matrix

**Your turn.** Build a complete one-hot encoding pipeline from scratch.

## 8.1 Task: Encode your own sentence

Modify the corpus below with your own text, then run the encoder:

In [ ]:
# TODO: Add your own sentences here
my_corpus = [
    "I love machine learning",
    "Python is great for AI",
    "NumPy arrays are fast"
]

# Build vocabulary
my_tokens = [word.lower() for sentence in my_corpus for word in sentence.split()]
my_vocab = sorted(set(my_tokens))
my_vocab_size = len(my_vocab)
my_word_to_idx = {word: idx for idx, word in enumerate(my_vocab)}

print(f"Vocabulary size: {my_vocab_size}")
print(f"Vocabulary: {my_vocab}")

# Encode a test sentence
test_sentence = "I love Python"
test_matrix = sentence_to_onehot_matrix(test_sentence, my_word_to_idx, my_vocab_size)

print(f"\nTest sentence: '{test_sentence}'")
print(f"Encoded matrix shape: {test_matrix.shape}")
print("\nMatrix:")
print(test_matrix)

## 8.2 Challenge: Batch encoding

Encode **all sentences** in your corpus at once, creating a 3D array:

In [ ]:
def encode_corpus(corpus: List[str], word_to_idx: Dict[str, int], vocab_size: int) -> np.ndarray:
    """
    Encode an entire corpus into a 3D array.
    
    Returns:
        Array of shape (num_sentences, max_length, vocab_size)
    """
    # Find max sentence length
    max_length = max(len(sentence.split()) for sentence in corpus)
    
    # Preallocate 3D array
    batch = np.zeros((len(corpus), max_length, vocab_size))
    
    # Fill in each sentence
    for i, sentence in enumerate(corpus):
        matrix = sentence_to_onehot_matrix(sentence, word_to_idx, vocab_size)
        batch[i, :matrix.shape[0], :] = matrix
    
    return batch

# Encode the full corpus
corpus_batch = encode_corpus(my_corpus, my_word_to_idx, my_vocab_size)
print(f"Corpus batch shape: {corpus_batch.shape}")
print(f"(num_sentences={len(my_corpus)}, max_length={corpus_batch.shape[1]}, vocab_size={my_vocab_size})")

## 8.3 Bonus: Connection to LLM embeddings

One-hot encoding is the **input** to classical NLP models. Modern LLMs take it further:

```
Text → Tokenizer → One-Hot → Embedding Layer → Transformer → Output
```

The **embedding layer** learns a dense transformation:

```python
# One-hot: sparse vector (vocab_size,) — e.g., (50000,)
# Embedding: dense vector (embed_dim,) — e.g., (768,)
```

This is the **compression** step that makes modern NLP tractable. We explore this in **Module 4: Dense Embeddings**.

---
# Step 9 · Summary & Key Takeaways

You learned:

1. **Why Python lists fail** — non-contiguous memory, pointer overhead, no vectorization
2. **NumPy arrays** — contiguous blocks, 50-100x faster, typed data
3. **The .shape property** — `(rows, columns)` or `(depth, rows, columns)`
4. **Rank-1 vs Rank-2** — `(5,)` ≠ `(5,1)` ≠ `(1,5)` — the shape bug
5. **Reshaping** — `.reshape()`, `.flatten()`, `.ravel()` — no data copy
6. **Indexing** — `arr[row, col]`, slicing, boolean masking
7. **One-hot encoding** — words → vocabulary → index → binary vectors → matrix
8. **The sparsity problem** — 99%+ zeros → need for dense embeddings

> ### 🧮 The one takeaway
>
> **Shape is everything.** Always check `.shape` first when debugging. The difference between `(n,)`, `(n,1)`, and `(1,n)` is the source of 90% of NumPy errors.

## What's next?

- **Module 4: Vectorization & Broadcasting** — Writing fast NumPy code without loops
- **Module 5: DataFrames & Series** — Pandas for tabular data
- **Module 6: Cleaning & Transformation** — Real-world data preprocessing

## Try it yourself

1. **Reshape practice** — Create a `(24,)` array and reshape it 5 different ways
2. **One-hot encoder** — Extend the encoder to handle unknown words (add an `<UNK>` token)
3. **Reverse encoding** — Write `onehot_to_word()` — given a one-hot vector, return the word
4. **Compare with scikit-learn** — Import `OneHotEncoder` from sklearn and compare results
5. **Visualize 3D arrays** — Create a 3D array and plot each "layer" as a separate heatmap

Happy array thinking! 🧮